In [1]:
import pandas as pd, numpy as np
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F  # F.col, F.count, F.max, F.upper ...

spark = (
    SparkSession.builder
        .appName("SparkML")
        .config("spark.sql.execution.arrow.pyspark.enabled", "true")  # pandas NaN -> Spark null
        .enableHiveSupport()
        .getOrCreate()
)

DATA_PATH = Path("./data")

26/10/19 04:25:44 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
titanic_sdf = spark.read.csv(str(DATA_PATH / "titanic_train.csv"), header=True, inferSchema=True)
titanic_sdf = titanic_sdf.cache()   
titanic_sdf.createOrReplaceTempView("titanic_view")

In [3]:
# SQL
spark.sql("""
select *
from titanic_view
""").show()

# DataFrame
titanic_sdf.select("*").show()

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL|       S|
|          6|       0|     3|    Moran, Mr. James|  male|NULL|    0|    0|      

In [4]:
# SQL
spark.sql("""
select *
from titanic_view
limit 10
""").show()

# DataFrame
titanic_sdf.select("*").limit(10).show()

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL|       S|
|          6|       0|     3|    Moran, Mr. James|  male|NULL|    0|    0|      

In [5]:
# SQL
spark.sql("""
select upper(name) as cap_name, a.*
from titanic_view as a
limit 10
""").show()

# DataFrame
titanic_sdf.select(F.upper(F.col("name")).alias("cap_name"), "*").limit(10).show()

+--------------------+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|            cap_name|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+--------------------+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|BRAUND, MR. OWEN ...|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|CUMINGS, MRS. JOH...|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|HEIKKINEN, MISS. ...|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
|FUTRELLE, MRS. JA...|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|
|ALLEN, MR. WILLIA...|          5|       0|     

In [6]:
# SQL
spark.sql("""
select a.*, upper(name) as cap_name
from titanic_view as a
limit 10
""").show()

# DataFrame
titanic_sdf.withColumn("cap_name", F.upper(F.col("name"))).limit(10).show()

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+--------------------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|            cap_name|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+--------------------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|BRAUND, MR. OWEN ...|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|CUMINGS, MRS. JOH...|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|HEIKKINEN, MISS. ...|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|FUTRELLE, MRS. JA...|
|          5|       0|     3|Allen, Mr. Willia..

In [7]:
# SQL
spark.sql("""
select a.*, substring(cabin, 0, 1) as cap_cabin
from titanic_view as a
limit 10
""").show()

# DataFrame
titanic_sdf.select("*", F.substring(F.col("Cabin"), 0, 1).alias("cap_cabin")).limit(10).show()
titanic_sdf.withColumn("cap_cabin", F.substring(F.col("Cabin"), 0, 1)).limit(10).show()

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+---------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|cap_cabin|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+---------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|     NULL|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|        C|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|     NULL|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S|        C|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL|       S|     NULL|


In [8]:
# SQL
spark.sql("""
select *
from titanic_view
where Pclass = 1 and Age > 60
limit 10
""").show()

# DataFrame
titanic_sdf.filter(
    (F.col("Pclass") == 1) & (F.col("Age") > 60)
).limit(10).show()

+-----------+--------+------+--------------------+------+----+-----+-----+--------+-------+-----------+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|  Ticket|   Fare|      Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+--------+-------+-----------+--------+
|         55|       0|     1|Ostby, Mr. Engelh...|  male|65.0|    0|    1|  113509|61.9792|        B30|       C|
|         97|       0|     1|Goldschmidt, Mr. ...|  male|71.0|    0|    0|PC 17754|34.6542|         A5|       C|
|        171|       0|     1|Van der hoef, Mr....|  male|61.0|    0|    0|  111240|   33.5|        B19|       S|
|        253|       0|     1|Stead, Mr. Willia...|  male|62.0|    0|    0|  113514|  26.55|        C87|       S|
|        276|       1|     1|Andrews, Miss. Ko...|female|63.0|    1|    0|   13502|77.9583|         D7|       S|
|        439|       0|     1|   Fortune, Mr. Mark|  male|64.0|    1|    4|   19950|  263.0|C23 C

In [9]:
# SQL
spark.sql("""
select pclass, name, age
from titanic_view
where upper(name) like "%MR.%"
limit 10
""").show()

# DataFrame
titanic_sdf.filter(
    F.upper(F.col("name")).like("%MR.%")
).select(F.col("pclass"), F.col("name"), F.col("age")).limit(10).show()

+------+--------------------+----+
|pclass|                name| age|
+------+--------------------+----+
|     3|Braund, Mr. Owen ...|22.0|
|     3|Allen, Mr. Willia...|35.0|
|     3|    Moran, Mr. James|NULL|
|     1|McCarthy, Mr. Tim...|54.0|
|     3|Saundercock, Mr. ...|20.0|
|     3|Andersson, Mr. An...|39.0|
|     2|Williams, Mr. Cha...|NULL|
|     2|Fynney, Mr. Joseph J|35.0|
|     2|Beesley, Mr. Lawr...|34.0|
|     1|Sloper, Mr. Willi...|28.0|
+------+--------------------+----+

+------+--------------------+----+
|pclass|                name| age|
+------+--------------------+----+
|     3|Braund, Mr. Owen ...|22.0|
|     3|Allen, Mr. Willia...|35.0|
|     3|    Moran, Mr. James|NULL|
|     1|McCarthy, Mr. Tim...|54.0|
|     3|Saundercock, Mr. ...|20.0|
|     3|Andersson, Mr. An...|39.0|
|     2|Williams, Mr. Cha...|NULL|
|     2|Fynney, Mr. Joseph J|35.0|
|     2|Beesley, Mr. Lawr...|34.0|
|     1|Sloper, Mr. Willi...|28.0|
+------+--------------------+----+



In [10]:
# SQL
spark.sql("""
select *
from titanic_view
where pclass in (1, 2)
limit 10
""").show()

# DataFrame
titanic_sdf.filter(
    F.col("pclass").isin(1, 2)
).limit(10).show()

+-----------+--------+------+--------------------+------+----+-----+-----+--------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|  Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+--------+-------+-----+--------+
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|PC 17599|71.2833|  C85|       C|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|  113803|   53.1| C123|       S|
|          7|       0|     1|McCarthy, Mr. Tim...|  male|54.0|    0|    0|   17463|51.8625|  E46|       S|
|         10|       1|     2|Nasser, Mrs. Nich...|female|14.0|    1|    0|  237736|30.0708| NULL|       C|
|         12|       1|     1|Bonnell, Miss. El...|female|58.0|    0|    0|  113783|  26.55| C103|       S|
|         16|       1|     2|Hewlett, Mrs. (Ma...|female|55.0|    0|    0|  248706|   16.0| NULL|       S|
|         18|       1|     2|Williams

In [11]:
# SQL
spark.sql("""
select *
from titanic_view
order by pclass
limit 10
""").show()

# DataFrame
titanic_sdf.orderBy(F.col("pclass")).limit(10).show()

+-----------+--------+------+--------------------+------+----+-----+-----+--------+--------+-----------+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|  Ticket|    Fare|      Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+--------+--------+-----------+--------+
|         35|       0|     1|Meyer, Mr. Edgar ...|  male|28.0|    1|    0|PC 17604| 82.1708|       NULL|       C|
|         89|       1|     1|Fortune, Miss. Ma...|female|23.0|    3|    2|   19950|   263.0|C23 C25 C27|       S|
|         36|       0|     1|Holverson, Mr. Al...|  male|42.0|    1|    0|  113789|    52.0|       NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|PC 17599| 71.2833|        C85|       C|
|         53|       1|     1|Harper, Mrs. Henr...|female|49.0|    1|    0|PC 17572| 76.7292|        D33|       C|
|          7|       0|     1|McCarthy, Mr. Tim...|  male|54.0|    0|    0|   17463| 51.8

In [12]:
# SQL
spark.sql("""
select *
from titanic_view
order by pclass asc, name desc
limit 10
""").show()

# DataFrame
titanic_sdf.orderBy(F.col("pclass").asc(), F.col("name").desc()).limit(10).show()
titanic_sdf.orderBy(F.col("pclass"), F.col("name"), ascending=[True, False]).limit(10).show()

+-----------+--------+------+--------------------+------+----+-----+-----+--------+--------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|  Ticket|    Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+--------+--------+-----+--------+
|        326|       1|     1|Young, Miss. Mari...|female|36.0|    0|    0|PC 17760|135.6333|  C32|       C|
|        556|       0|     1|  Wright, Mr. George|  male|62.0|    0|    0|  113807|   26.55| NULL|       S|
|         56|       1|     1|   Woolner, Mr. Hugh|  male|NULL|    0|    0|   19947|    35.5|  C52|       S|
|        352|       0|     1|Williams-Lambert,...|  male|NULL|    0|    0|  113510|    35.0| C128|       S|
|        156|       0|     1|Williams, Mr. Cha...|  male|51.0|    0|    1|PC 17597| 61.3792| NULL|       C|
|        378|       0|     1|Widener, Mr. Harr...|  male|27.0|    0|    2|  113503|   211.5|  C82|       C|
|        857|       1|     1

+-----------+--------+------+--------------------+------+----+-----+-----+--------+--------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|  Ticket|    Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+--------+--------+-----+--------+
|        326|       1|     1|Young, Miss. Mari...|female|36.0|    0|    0|PC 17760|135.6333|  C32|       C|
|        556|       0|     1|  Wright, Mr. George|  male|62.0|    0|    0|  113807|   26.55| NULL|       S|
|         56|       1|     1|   Woolner, Mr. Hugh|  male|NULL|    0|    0|   19947|    35.5|  C52|       S|
|        352|       0|     1|Williams-Lambert,...|  male|NULL|    0|    0|  113510|    35.0| C128|       S|
|        156|       0|     1|Williams, Mr. Cha...|  male|51.0|    0|    1|PC 17597| 61.3792| NULL|       C|
|        378|       0|     1|Widener, Mr. Harr...|  male|27.0|    0|    2|  113503|   211.5|  C82|       C|
|        857|       1|     1

In [13]:
# SQL
spark.sql("""
select count(*) as cnt
from titanic_view
""").show()

# DataFrame
titanic_sdf.select(F.count("*").alias("cnt")).show()

+---+
|cnt|
+---+
|891|
+---+

+---+
|cnt|
+---+
|891|
+---+



In [14]:
# SQL
spark.sql("""
select max(age), min(age), avg(sibsp)
from titanic_view
""").show()

# DataFrame
titanic_sdf.select(F.max(F.col("age")), F.min(F.col("age")), F.avg(F.col("sibsp"))).show()

+--------+--------+------------------+
|max(age)|min(age)|        avg(sibsp)|
+--------+--------+------------------+
|    80.0|    0.42|0.5230078563411896|
+--------+--------+------------------+



[Stage 35:>                                                         (0 + 1) / 1]

+--------+--------+------------------+
|max(age)|min(age)|        avg(sibsp)|
+--------+--------+------------------+
|    80.0|    0.42|0.5230078563411896|
+--------+--------+------------------+



In [15]:
# SQL
spark.sql("""
select pclass, max(age), min(age), avg(sibsp)
from titanic_view
group by pclass
""").show()

# DataFrame
titanic_sdf.groupBy(F.col("pclass")).agg(
    F.max(F.col("age")),
    F.min(F.col("age")),
    F.avg(F.col("sibsp"))
).show()

+------+--------+--------+-------------------+
|pclass|max(age)|min(age)|         avg(sibsp)|
+------+--------+--------+-------------------+
|     1|    80.0|    0.92| 0.4166666666666667|
|     3|    74.0|    0.42|  0.615071283095723|
|     2|    70.0|    0.67|0.40217391304347827|
+------+--------+--------+-------------------+

+------+--------+--------+-------------------+
|pclass|max(age)|min(age)|         avg(sibsp)|
+------+--------+--------+-------------------+
|     1|    80.0|    0.92| 0.4166666666666667|
|     3|    74.0|    0.42|  0.615071283095723|
|     2|    70.0|    0.67|0.40217391304347827|
+------+--------+--------+-------------------+



In [16]:
# SQL
spark.sql("""
select
    pclass,
    max(age) as max_age,
    min(age) as min_age,
    avg(sibsp) as avg_sibsp
from titanic_view
group by pclass
having max(age) > 70
""").show()

# DataFrame
titanic_sdf.groupBy(F.col("pclass")).agg(
    F.max(F.col("age")).alias("max_age"),
    F.min(F.col("age")).alias("min_age"),
    F.avg(F.col("sibsp")).alias("avg_sibsp")
).filter(
    F.col("max_age") > 70
).show()

+------+-------+-------+------------------+
|pclass|max_age|min_age|         avg_sibsp|
+------+-------+-------+------------------+
|     1|   80.0|   0.92|0.4166666666666667|
|     3|   74.0|   0.42| 0.615071283095723|
+------+-------+-------+------------------+

+------+-------+-------+------------------+
|pclass|max_age|min_age|         avg_sibsp|
+------+-------+-------+------------------+
|     1|   80.0|   0.92|0.4166666666666667|
|     3|   74.0|   0.42| 0.615071283095723|
+------+-------+-------+------------------+



In [17]:
# SQL
spark.sql("""
select *
from titanic_view
where age is null and cabin is not null
limit 10
""").show()

# DataFrame
titanic_sdf.filter(
    F.col("Age").isNull() & F.col("Cabin").isNotNull()
).limit(10).show()

+-----------+--------+------+--------------------+------+----+-----+-----+--------+--------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|  Ticket|    Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+--------+--------+-----+--------+
|         32|       1|     1|Spencer, Mrs. Wil...|female|NULL|    1|    0|PC 17569|146.5208|  B78|       C|
|         56|       1|     1|   Woolner, Mr. Hugh|  male|NULL|    0|    0|   19947|    35.5|  C52|       S|
|        129|       1|     3|   Peter, Miss. Anna|female|NULL|    1|    1|    2668| 22.3583|F E69|       C|
|        167|       1|     1|Chibnall, Mrs. (E...|female|NULL|    0|    1|  113505|    55.0|  E33|       S|
|        186|       0|     1|Rood, Mr. Hugh Ro...|  male|NULL|    0|    0|  113767|    50.0|  A32|       S|
|        285|       0|     1|Smith, Mr. Richar...|  male|NULL|    0|    0|  113056|    26.0|  A19|       S|
|        299|       1|     1

In [18]:
# SQL
spark.sql("""
select a.*,
    case
        when age is null then "NA"
        when age <= 6 then "baby"
        when age <= 12 then "child"
        when age <= 18 then "teenage"
        when age <= 25 then "student"
        when age <= 35 then "young_adult"
        when age <= 60 then "adult"
        else "elderly"
    end as Age_Category
from titanic_view as a
limit 10
""").show()

# DataFrame
titanic_sdf.withColumn(
    "Age_Category",
    F.when(F.col("Age").isNull(), "NA")
     .when(F.col("Age") <= 6, "baby")
     .when(F.col("Age") <= 12, "child")
     .when(F.col("Age") <= 18, "teenage")
     .when(F.col("Age") <= 25, "student")
     .when(F.col("Age") <= 35, "young_adult")
     .when(F.col("Age") <= 60, "adult")
     .otherwise("elderly")
).limit(10).show()

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+------------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|Age_Category|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+------------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|     student|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|       adult|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S| young_adult|
|          4|       1|     1|Futrelle, Mrs. Ja...|female|35.0|    1|    0|          113803|   53.1| C123|       S| young_adult|
|          5|       0|     3|Allen, Mr. Willia...|  male|35.0|    0|    0|          373450|   8.05| NULL